# Week 68, Day 5: When to extend and when not to

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w068d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def overall(hot_fraction, local_speedup):
    return round(1 / ((1 - hot_fraction) + hot_fraction / local_speedup), 2)

print(overall(0.5, 100))
print(overall(0.9, 100))
print(overall(0.9, 10))

In [ ]:
LADDER = [
    ("measure", lambda p: not p["profiled"]),
    ("better algorithm", lambda p: p["complexity"] in ("quadratic", "cubic")),
    ("NumPy or a library", lambda p: p["numeric_arrays"]),
    ("caching", lambda p: p["repeated_inputs"]),
    ("concurrency", lambda p: p["mostly_waiting"]),
    ("PyPy, Numba or mypyc", lambda p: p["pure_python_loops"] and not p["needs_cpython_only_libraries"]),
]

def next_step(problem):
    for step, applies in LADDER:
        if applies(problem):
            return step
    return "write an extension"

base = dict(profiled=True, complexity="linear", numeric_arrays=False, repeated_inputs=False,
            mostly_waiting=False, pure_python_loops=True, needs_cpython_only_libraries=True)
print(next_step(dict(base, profiled=False)), "|", next_step(dict(base, complexity="quadratic")), "|", next_step(base))

In [ ]:
def overall(f, s):
    return 1 / ((1 - f) + f / s)

def saved_hours_per_year(runs_per_day, seconds_before, seconds_after, days=365):
    return runs_per_day * days * (seconds_before - seconds_after) / 3600

def break_even_years(build_hours, upkeep_hours_per_year, saved_per_year):
    net = saved_per_year - upkeep_hours_per_year
    return None if net <= 0 else round(build_hours / net, 2)

before = 40.0
after = before / overall(0.8, 20)
saved = saved_hours_per_year(50, before, after)
print(round(after, 1), "seconds per run;", round(saved), "hours a year;", break_even_years(40, 20, saved), "years to pay back")
print(break_even_years(40, 20, saved_hours_per_year(1 / 30, before, after)))

In [ ]:
def total_cost(n, batch, call_overhead=200, per_item=5):
    calls = -(-n // batch)
    return calls * call_overhead + n * per_item

n = 1_000_000
for batch in (1, 10, 1000, n):
    print(f"batch {batch:>7}: {total_cost(n, batch) / 1e6:8.2f} ms (nanosecond units)")

In [ ]:
def parse_wheel(filename):
    parts = filename[:-len(".whl")].split("-")
    name, version = parts[0], parts[1]
    python, abi, platform = parts[-3:]
    return {"name": name, "version": version, "python": python, "abi": abi, "platform": platform}

for f in ["requests-2.32.3-py3-none-any.whl",
          "numpy-2.1.0-cp313-cp313-manylinux_2_17_x86_64.whl",
          "fastjson-1.0-cp39-abi3-win_amd64.whl"]:
    w = parse_wheel(f)
    kind = "pure Python" if w["abi"] == "none" else "stable ABI" if w["abi"] == "abi3" else "one Python version"
    print(f"{w['name']:9} {w['python']:5} {w['abi']:6} {w['platform']:24} {kind}")

In [ ]:
def suggest(existing_c_library, numeric_loop, large_component, team_knows_rust, needs_internals):
    if existing_c_library:
        return "ctypes or cffi"
    if needs_internals:
        return "the C API"
    if large_component and team_knows_rust:
        return "Rust with PyO3"
    if numeric_loop:
        return "Cython (or Numba)"
    return "Cython, or reconsider: maybe mypyc"

print(suggest(True, False, False, False, False), "|", suggest(False, True, False, False, False), "|",
      suggest(False, False, True, True, False))

> **Key idea.** Climb the ladder before extending: measure, improve the algorithm, use fast libraries such as NumPy, cache, use concurrency for waiting, try PyPy, Numba or mypyc, and only then write an extension. Amdahl's law limits the overall gain to 1 / (1 - f) for a hot fraction f; check that the saved time (or latency that matters) outweighs building and maintaining the extension. Each call across the Python-native boundary has a fixed overhead, so batch whole arrays into one call and avoid copies with the buffer protocol. Compiled code ships as wheels per platform, named name-version-python-abi-platform (abi3 for the stable ABI, manylinux and musllinux for Linux), built in CI. Use ctypes or cffi for existing C libraries, Cython for numeric loops, Rust with PyO3 for larger safe components, and the raw C API only for full control.

### Exercise: What the hot spot is worth

Write `overall_speedup(hot_fraction, local_speedup)`: Amdahl's overall speed-up, rounded to 2 places; `ceiling(hot_fraction)`: the most it could ever be, rounded to 2 places (`float("inf")` when the fraction is 1); and `local_needed(hot_fraction, target)`: the local speed-up the hot part needs for the program to reach `target` overall, rounded to 2 places, or `None` if the target is at or above the ceiling.

In [ ]:
def overall_speedup(hot_fraction, local_speedup):
    ___

def ceiling(hot_fraction):
    ___

def local_needed(hot_fraction, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_amdahl():\n    \"Amdahl's law\"\n    assert overall_speedup(0.5, 100) == 1.98 and overall_speedup(0.9, 10) == 5.26 and overall_speedup(1.0, 8) == 8.0\n    assert ceiling(0.9) == 10.0 and ceiling(0.5) == 2.0 and ceiling(1.0) == float(\"inf\")\n    assert local_needed(0.9, 5) == 9.0 and local_needed(0.8, 4) == 16.0 and local_needed(0.5, 2) is None\n    assert local_needed(1.0, 50) == 50.0")

### Exercise: Does it pay for itself?

Write `hours_saved(runs_per_day, seconds_before, seconds_after, days=365)`: the hours of waiting saved per year, rounded to 1 place; `payback_years(build_hours, upkeep_per_year, saved_per_year)`: years until the saved time covers the build, rounded to 2 places, or `None` if the upkeep eats all the savings; and `worth_it(job, horizon_years=3)`: True if a job dict (with `runs_per_day`, `before`, `after`, `build_hours` and `upkeep`) pays back within the horizon.

In [ ]:
def hours_saved(runs_per_day, seconds_before, seconds_after, days=365):
    ___

def payback_years(build_hours, upkeep_per_year, saved_per_year):
    ___

def worth_it(job, horizon_years=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_breakeven():\n    \"time saved versus time spent\"\n    assert hours_saved(50, 40, 10) == 152.1 and hours_saved(1, 3600, 0, days=10) == 10.0\n    assert payback_years(40, 20, 152.1) == 0.3 and payback_years(40, 50, 30) is None\n    nightly = {\"runs_per_day\": 1, \"before\": 1800, \"after\": 1500, \"build_hours\": 100, \"upkeep\": 10}\n    busy = {\"runs_per_day\": 200, \"before\": 2.0, \"after\": 0.2, \"build_hours\": 80, \"upkeep\": 5}\n    assert not worth_it(nightly) and worth_it(busy) and not worth_it(dict(busy, runs_per_day=5))")

### Exercise: The cost of crossing

Using the model `calls * call_overhead + n * per_item` (with `calls` the number of batches, rounding up), write `cost(n, batch, call_overhead, per_item)`; `best_batch(n, call_overhead, per_item, max_batch)`: the batch size from 1 up to `max_batch` with the lowest cost (the smallest such size on ties); and `python_loop_gain(n, python_per_item, native_per_item, call_overhead)`: how many times faster calling a native function **once per item** is than doing the item in Python (rounded to 2 places; below 1 means it's slower).

In [ ]:
def cost(n, batch, call_overhead, per_item):
    ___

def best_batch(n, call_overhead, per_item, max_batch):
    ___

def python_loop_gain(n, python_per_item, native_per_item, call_overhead):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_boundary():\n    \"batching across the boundary\"\n    assert cost(1000, 1, 200, 5) == 205000 and cost(1000, 1000, 200, 5) == 5200 and cost(1001, 1000, 200, 5) == 5405\n    assert best_batch(1000, 200, 5, 64) == 63\n    assert best_batch(100, 10, 1, 1000) == 100 and best_batch(5, 0, 1, 3) == 1\n    assert python_loop_gain(1000, 50, 1, 200) == 0.25 and python_loop_gain(1000, 500, 1, 100) == 4.95")

### Exercise: Reading wheel names

Write `parse_wheel(filename)`: a dict with `name`, `version`, `build` (the optional build tag, or `None`), `python`, `abi` and `platform`, from a name like `fastjson-1.0-2-cp39-abi3-win_amd64.whl` (the build tag is present when there are six dash-separated parts). Then `installable(filename, python, platform, glibc=(2, 28))` for a system running CPython `python` (a tag like `"cp313"`) on `platform` (like `"linux_x86_64"`, `"win_amd64"`): the python tag must be `py3`, or equal to it, or, when the ABI is `abi3`, a `cp3X` tag no newer than it; the ABI must be `none`, `abi3` or equal to the python tag; the platform must be `any`, equal, or, for Linux, a `manylinux_A_B_arch` tag with the same architecture and `(A, B)` no newer than `glibc`. Compare versions as numbers, not strings.

In [ ]:
def parse_wheel(filename):
    ___

def installable(filename, python, platform, glibc=(2, 28)):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wheels():\n    \"wheel tags\"\n    assert parse_wheel(\"fastjson-1.0-2-cp39-abi3-win_amd64.whl\") == {\n        \"name\": \"fastjson\", \"version\": \"1.0\", \"build\": \"2\", \"python\": \"cp39\", \"abi\": \"abi3\", \"platform\": \"win_amd64\"}\n    assert parse_wheel(\"requests-2.32.3-py3-none-any.whl\")[\"build\"] is None\n    assert installable(\"requests-2.32.3-py3-none-any.whl\", \"cp313\", \"win_amd64\")\n    assert installable(\"numpy-2.1.0-cp313-cp313-manylinux_2_17_x86_64.whl\", \"cp313\", \"linux_x86_64\")\n    assert not installable(\"numpy-2.1.0-cp313-cp313-manylinux_2_17_x86_64.whl\", \"cp312\", \"linux_x86_64\")\n    assert installable(\"fast-1.0-cp39-abi3-manylinux_2_17_x86_64.whl\", \"cp313\", \"linux_x86_64\")\n    assert not installable(\"fast-1.0-cp313-abi3-manylinux_2_17_x86_64.whl\", \"cp39\", \"linux_x86_64\")\n    assert not installable(\"new-1.0-cp313-cp313-manylinux_2_34_x86_64.whl\", \"cp313\", \"linux_x86_64\")\n    assert not installable(\"arm-1.0-cp313-cp313-manylinux_2_17_aarch64.whl\", \"cp313\", \"linux_x86_64\")\n    assert not installable(\"win-1.0-cp313-cp313-win_amd64.whl\", \"cp313\", \"linux_x86_64\")")

### Exercise: The next rung

Write `next_step(problem)` for a dict of facts about a slow program, returning the first rung that applies: `"measure"` if it hasn't been `profiled`; `"better algorithm"` if its `complexity` is `"quadratic"` or worse (`"cubic"`, `"exponential"`); `"use a library"` if it's `numeric_arrays` work; `"cache"` if it has `repeated_inputs`; `"concurrency"` if it's `mostly_waiting`; `"try a faster Python"` if it's `pure_python` code and doesn't `need_cpython`; otherwise `"write an extension"`. Then `steps_before_extension(problem)`: how many rungs were skipped as not applying before reaching the answer.

In [ ]:
def next_step(problem):
    ___

def steps_before_extension(problem):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ladder():\n    \"the ladder of alternatives\"\n    base = {\"profiled\": True, \"complexity\": \"linear\", \"numeric_arrays\": False, \"repeated_inputs\": False,\n            \"mostly_waiting\": False, \"pure_python\": True, \"need_cpython\": True}\n    assert next_step(dict(base, profiled=False)) == \"measure\"\n    assert next_step(dict(base, complexity=\"cubic\")) == \"better algorithm\"\n    assert next_step(dict(base, numeric_arrays=True)) == \"use a library\"\n    assert next_step(dict(base, mostly_waiting=True)) == \"concurrency\"\n    assert next_step(dict(base, need_cpython=False)) == \"try a faster Python\"\n    assert next_step(base) == \"write an extension\" and steps_before_extension(base) == 6\n    assert steps_before_extension(dict(base, repeated_inputs=True)) == 3")

### Bug hunt: Amdahl misapplied

The estimate assumes the whole program gets the hot spot's local speed-up, so it promises `20` times faster. Only 60% of the time is in that part: apply Amdahl's law (rounding to 2 places) so this prints `2.33`.

In [ ]:
hot_fraction = 0.6
local_speedup = 20
print(local_speedup)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Amdahl's law\"\n    expect_output(\"2.33\")")

### Bug hunt: Versions compared as text

The check compares Python version tags as **strings**, and `"cp39" > "cp313"` because `"9"` comes after `"1"`, so it wrongly says a wheel for 3.9 and later can't be installed on 3.13: this prints `False`. Compare the minor version numbers as integers so it prints `True`.

In [ ]:
wheel_tag, python_tag = "cp39", "cp313"
print(wheel_tag <= python_tag)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"numbers, not strings\"\n    expect_output(\"True\")")

### Bug hunt: One call per item

The code calls a "native" function once per item, paying the fixed boundary overhead a thousand times, so the total cost is `205000` units. Pass all the items in **one** call so the overhead is paid once and this prints `5200`.

In [ ]:
CALL_OVERHEAD, PER_ITEM = 200, 5

def native_sum(items):
    return CALL_OVERHEAD + PER_ITEM * len(items)     # the cost of one call

items = list(range(1000))
total_cost = sum(native_sum([x]) for x in items)
print(total_cost)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"batch the calls\"\n    expect_output(\"5200\")")

### Puzzle: Amdahl's law

Arrange the lines of a function that computes the overall speed-up from a hot fraction and its local speed-up, refusing impossible fractions. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the ceiling on overall speed-up when 75% of the time is in the hot spot
q = ___   # the ABI tag of a pure-Python wheel
r = ___   # True or False: "cp39" < "cp313" compared as strings
s = ___   # cost of 10 items in one call with overhead 100 and 2 per item
t = ___   # the first rung of the ladder

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == \"none\" and r is False, \"p, q and r\"\n    assert s == 120 and t == \"measure\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an options report

Help decide how to speed up a program. Write `evaluate(hot_fraction, run_seconds, runs_per_year, options)` where each option is a dict with `name`, `local_speedup`, `build_hours` and `upkeep_hours` (per year). For each, compute:

- `overall`: Amdahl's overall speed-up, rounded to 2 places;
- `saved_hours`: hours saved per year, `runs_per_year * run_seconds * (1 - 1 / overall) / 3600`, rounded to 1 place;
- `payback_years`: `build_hours / (saved_hours - upkeep_hours)` rounded to 2 places, or `None` if that isn't positive.

Return a list of dicts (the option's name plus those three keys), in the order given. Then `recommend(hot_fraction, run_seconds, runs_per_year, options, target_speedup, horizon_years=2)`: among the options that reach `target_speedup` overall **and** pay back within the horizon, the one with the fewest `build_hours` (ties: the higher overall speed-up, then the order given), or `None`.

In [ ]:
def evaluate(hot_fraction, run_seconds, runs_per_year, options):
    ___

def recommend(hot_fraction, run_seconds, runs_per_year, options, target_speedup, horizon_years=2):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"choosing an option\"\n    options = [\n        {\"name\": \"NumPy rewrite\", \"local_speedup\": 30, \"build_hours\": 16, \"upkeep_hours\": 2},\n        {\"name\": \"Cython\", \"local_speedup\": 50, \"build_hours\": 40, \"upkeep_hours\": 10},\n        {\"name\": \"Rust\", \"local_speedup\": 80, \"build_hours\": 120, \"upkeep_hours\": 15},\n        {\"name\": \"caching\", \"local_speedup\": 2, \"build_hours\": 4, \"upkeep_hours\": 1},\n    ]\n    report = evaluate(0.9, 60, 20000, options)\n    assert [r[\"name\"] for r in report] == [\"NumPy rewrite\", \"Cython\", \"Rust\", \"caching\"]\n    assert report[0] == {\"name\": \"NumPy rewrite\", \"overall\": 7.69, \"saved_hours\": 290.0, \"payback_years\": 0.06}\n    assert report[3][\"overall\"] == 1.82 and report[2][\"payback_years\"] == round(120 / (report[2][\"saved_hours\"] - 15), 2)\n    assert recommend(0.9, 60, 20000, options, target_speedup=7) == \"NumPy rewrite\"\n    assert recommend(0.9, 60, 20000, options, target_speedup=8.4) == \"Cython\"\n    assert recommend(0.9, 60, 20000, options, target_speedup=8.5) == \"Rust\"\n    assert recommend(0.9, 60, 20000, options, target_speedup=12) is None\n    assert recommend(0.9, 60, 20, options, target_speedup=1.5) is None")

## Warm-up

Making half the program a hundred times faster speeds the whole thing up by less than 2. If 90% of the time is in the code you speed up, a tenfold local gain gives about 5 overall and a hundredfold gives about 9. Amdahl's law (Week 66) is the first check on any plan to rewrite part of a program in a faster language.

## The ladder

Before writing an extension, climb the cheaper rungs first. Each is less work and less risk than the one after it, and each often removes the need for the next:

1. **Measure.** Profile (Week 65) and confirm where the time goes. Decide what "fast enough" means.
2. **Better algorithm or data structure.** An O(n²) to O(n log n) change beats any language switch (Week 65 Day 2).
3. **Use what's already fast**: built-ins, the standard library, and libraries such as **NumPy**, pandas or Polars that already run in compiled code (Week 65 Day 5).
4. **Avoid repeated work**: caching (Week 65 Day 4), batching I/O, doing less.
5. **Concurrency** if the time is waiting rather than computing (Weeks 66 and 67).
6. **A faster Python**: **PyPy** (a different interpreter with a just-in-time compiler, often several times faster for pure-Python loops), **Numba** (a decorator that compiles numeric functions to machine code), or **mypyc** (compiles type-annotated Python modules to C extensions).
7. **Write an extension**: Cython for numeric loops close to Python, Rust with PyO3 for larger, safe components, `ctypes`/`cffi` to call an existing C library, the raw C API only when you need full control.

Most performance problems are solved by step 2 or 3.

## What a speed-up is worth

Two pieces of arithmetic belong in every decision:

- **Amdahl's law**: if the part you speed up takes a fraction `f` of the run time and becomes `s` times faster, the whole program becomes `1 / ((1 - f) + f / s)` times faster. The ceiling is `1 / (1 - f)`. Profile to find `f`.
- **Break-even**: the time saved has to pay for the time spent. A job that runs 50 times a day and becomes 30 seconds faster saves about 150 hours a year of waiting (or of machine time, which costs money in the cloud); if the extension takes 40 hours to write and 20 hours a year to maintain, it pays for itself in about four months. A job that runs once a month saving 30 seconds never does.

And value isn't only hours: a response that drops from 2 seconds to 200 milliseconds may change whether a product is usable at all, while making a nightly batch job finish at 02:10 instead of 02:40 may change nothing for anyone.

## Crossing the boundary

Every call from Python into native code has a fixed **overhead**: converting arguments (Monday's `PyArg_ParseTuple`, or PyO3's conversions), checking types, possibly copying data (a list becomes a `Vec`), and converting the result back. That's typically tens to hundreds of nanoseconds, comparable to the cost of a whole Python operation. So:

- **Calling a fast native function once per element** from a Python loop often gains little or nothing: the loop and the call overhead dominate.
- **Batch**: pass a whole array, list or file to one native call and loop on the native side. That's why NumPy's whole-array operations are fast and NumPy-in-a-Python-loop isn't.
- **Avoid copies**: the buffer protocol (Wednesday) lets native code read Python's memory directly instead of converting each element.

The cost model: `n` items handled in calls of `b` items each cost about `(n / b) * call_overhead + n * per_item_work`. Larger batches cut the first term, until memory or latency limits how big a batch can be.

## Shipping compiled code

Pure-Python packages work everywhere. Compiled extensions must be **built for each platform**, and users without a compiler need a **wheel** that matches their system. A wheel's file name says what it's for: `name-version(-build)?-python-abi-platform.whl` (PEP 427):

- **python tag**: `py3` (any Python 3), `cp313` (CPython 3.13);
- **ABI tag**: `none` (pure Python), `cp313` (built against CPython 3.13's full ABI, only for that version), `abi3` (the stable ABI: CPython 3.x and later, from the version in the python tag);
- **platform tag**: `any`, `win_amd64`, `macosx_11_0_arm64`, `manylinux_2_17_x86_64` (works on Linux with glibc 2.17 or newer), `musllinux_1_2_x86_64` (Alpine and other musl systems).

So a project with a C or Rust extension typically publishes a dozen or more wheels per release, built automatically in continuous integration (with the free `cibuildwheel` or `maturin-action`), plus a source distribution for everything else. That's real, ongoing work, and part of the cost of extending.

## Choosing a tool

When an extension is justified, the tools compare like this:

| | good for | you write | safety | build needs |
|---|---|---|---|---|
| `ctypes` / `cffi` | calling an existing C library | Python (plus C declarations for cffi) | crashes on mistakes | nothing (ctypes); a compiler for cffi's API mode |
| Cython | numeric loops, gradual speed-ups of Python code | Python with C types | mostly safe; C rules for typed code | a C compiler |
| Rust + PyO3 | larger components, parsers, anything safety-critical | Rust | memory-safe, checked by the compiler | Rust toolchain and maturin |
| C API | full control, CPython internals | C | all of C's risks | a C compiler |
| Numba / mypyc | numeric functions / typed Python modules | Python | safe | the tool itself |

Questions that decide in practice: Does a library already do it? Who will maintain it, and do they know the language? Must it work on PyPy or many Python versions (favour `cffi`, the stable ABI, or pure Python)? Does it need to release the GIL for parallelism (Cython's `nogil` or Rust's `allow_threads`)?

## Practice

## Quiz

## Daily challenge

## Before you go

Pick the slowest program you've written in this course and climb the ladder with it on paper: which rung would you stop at? Tomorrow is the Forge: wrapping a C library properly and generating its stubs.